# 06 - Análise e Visualização

## Objetivo

Consumir o modelo dimensional Gold para produzir indicadores e visualizações sobre os registros de benefícios classificados como afastamento.

As análises utilizam exclusivamente as tabelas do modelo estrela:

- `fato_afastamentos`;
- `dim_tempo`;
- `dim_cid`;
- `dim_especie`;
- `dim_geografia`;
- `dim_atividade`.

O notebook não representa pessoas ou trabalhadores únicos.

O grão da tabela fato é:

> Uma linha representa um registro de benefício classificado como afastamento.

As análises geográficas representam município e unidade federativa de residência.

As análises por CNAE consideram somente os registros com atividade econômica informada. A ausência de CNAE não indica ausência de trabalho ou atividade profissional.

Os resultados representam contagens e distribuições internas da base. Não representam incidência, prevalência ou taxas populacionais.

## 1. Importações

As funções são centralizadas em uma única célula.

Nesta etapa serão utilizadas funções para:

- agregação;
- cálculo de percentuais;
- arredondamento;
- validação de relacionamentos;
- ordenação dos resultados.

In [0]:
# ============================================================
# IMPORTACOES
# ============================================================

from pyspark.sql.functions import (
    col,
    lit,
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    round as spark_round,
    when,
    desc
)

## 2. Parâmetros

Os nomes das tabelas são centralizados para facilitar manutenção e reutilização do notebook em outros ambientes.

In [0]:
# ============================================================
# PARAMETROS
# ============================================================

TABELA_FATO = (
    "afastamento_inss.gold.fato_afastamentos"
)

TABELA_DIM_TEMPO = (
    "afastamento_inss.gold.dim_tempo"
)

TABELA_DIM_CID = (
    "afastamento_inss.gold.dim_cid"
)

TABELA_DIM_ESPECIE = (
    "afastamento_inss.gold.dim_especie"
)

TABELA_DIM_GEOGRAFIA = (
    "afastamento_inss.gold.dim_geografia"
)

TABELA_DIM_ATIVIDADE = (
    "afastamento_inss.gold.dim_atividade"
)

print(f"Fato              : {TABELA_FATO}")
print(f"Dimensão tempo    : {TABELA_DIM_TEMPO}")
print(f"Dimensão CID      : {TABELA_DIM_CID}")
print(f"Dimensão espécie  : {TABELA_DIM_ESPECIE}")
print(f"Dimensão geografia: {TABELA_DIM_GEOGRAFIA}")
print(f"Dimensão atividade: {TABELA_DIM_ATIVIDADE}")

## 3. Leitura do modelo estrela

Cada tabela é carregada uma única vez.

A tabela fato e as dimensões serão armazenadas temporariamente em cache porque serão reutilizadas em diversas consultas analíticas.

In [0]:
# ============================================================
# LEITURA DO MODELO ESTRELA
# ============================================================

df_fato = (
    spark
    .table(TABELA_FATO)
)

df_dim_tempo = (
    spark
    .table(TABELA_DIM_TEMPO)
)

df_dim_cid = (
    spark
    .table(TABELA_DIM_CID)
)

df_dim_especie = (
    spark
    .table(TABELA_DIM_ESPECIE)
)

df_dim_geografia = (
    spark
    .table(TABELA_DIM_GEOGRAFIA)
)

df_dim_atividade = (
    spark
    .table(TABELA_DIM_ATIVIDADE)
)

total_registros_fato = df_fato.count()

print("=" * 65)
print("LEITURA DO MODELO ESTRELA")
print("=" * 65)
print(
    f"{'Fato de afastamentos':40}"
    f"{total_registros_fato:>15,}"
)
print(
    f"{'Dimensão tempo':40}"
    f"{df_dim_tempo.count():>15,}"
)
print(
    f"{'Dimensão CID':40}"
    f"{df_dim_cid.count():>15,}"
)
print(
    f"{'Dimensão espécie':40}"
    f"{df_dim_especie.count():>15,}"
)
print(
    f"{'Dimensão geografia':40}"
    f"{df_dim_geografia.count():>15,}"
)
print(
    f"{'Dimensão atividade':40}"
    f"{df_dim_atividade.count():>15,}"
)
print("=" * 65)

## 4. Validação do contrato analítico

Antes de calcular indicadores, o notebook verifica se a tabela fato contém as chaves, medidas e atributos necessários.

A análise será interrompida se alguma coluna obrigatória estiver ausente.

In [0]:
COLUNAS_OBRIGATORIAS_FATO = [
    "fato_sk",
    "tempo_sk",
    "cid_sk",
    "especie_sk",
    "geografia_sk",
    "atividade_sk",
    "qtd_beneficios",
    "ind_cid_informado",
    "ind_saude_mental",
    "ind_osteomuscular",
    "ind_cardiovascular",
    "ind_respiratorio",
    "ind_acidentario",
    "ind_afastamento_saude_mental",
    "ind_afastamento_osteomuscular",
    "ind_afastamento_cardiovascular",
    "ind_afastamento_respiratorio",
    "duracao_beneficio_dias",
    "idade_na_competencia",
    "faixa_etaria",
    "sexo",
    "clientela",
    "forma_filiacao",
    "qt_sm_rmi"
]

colunas_ausentes = [
    nome
    for nome in COLUNAS_OBRIGATORIAS_FATO
    if nome not in df_fato.columns
]

if colunas_ausentes:
    raise ValueError(
        "Colunas obrigatorias ausentes na fato: "
        + ", ".join(colunas_ausentes)
    )

print("Contrato analitico validado.")
print(
    f"Colunas obrigatorias encontradas: "
    f"{len(COLUNAS_OBRIGATORIAS_FATO)}"
)

In [0]:
resumo_validacao = (
    df_fato
    .agg(
        count(lit(1)).alias("total_registros"),
        countDistinct("fato_sk").alias(
            "fato_sk_distintas"
        ),
        spark_sum("qtd_beneficios").alias(
            "qtd_beneficios"
        ),
        spark_sum("ind_saude_mental").alias(
            "qtd_saude_mental"
        ),
        spark_sum("ind_osteomuscular").alias(
            "qtd_osteomuscular"
        ),
        spark_sum("ind_cardiovascular").alias(
            "qtd_cardiovascular"
        ),
        spark_sum("ind_respiratorio").alias(
            "qtd_respiratorio"
        ),
        spark_sum("ind_acidentario").alias(
            "qtd_acidentario"
        )
    )
    .first()
)

total_registros = resumo_validacao["total_registros"]
fato_sk_distintas = resumo_validacao["fato_sk_distintas"]
qtd_beneficios = resumo_validacao["qtd_beneficios"]
qtd_saude_mental = resumo_validacao["qtd_saude_mental"]
qtd_osteomuscular = resumo_validacao["qtd_osteomuscular"]
qtd_cardiovascular = resumo_validacao["qtd_cardiovascular"]
qtd_respiratorio = resumo_validacao["qtd_respiratorio"]
qtd_acidentario = resumo_validacao["qtd_acidentario"]

print("=" * 70)
print("VALIDACAO INICIAL DO MODELO ANALITICO")
print("=" * 70)
print(
    f"{'Total de registros':45}"
    f"{total_registros:>15,}"
)
print(
    f"{'Chaves fato_sk distintas':45}"
    f"{fato_sk_distintas:>15,}"
)
print(
    f"{'Soma de qtd_beneficios':45}"
    f"{qtd_beneficios:>15,}"
)
print(
    f"{'Saude mental':45}"
    f"{qtd_saude_mental:>15,}"
)
print(
    f"{'Osteomuscular':45}"
    f"{qtd_osteomuscular:>15,}"
)
print(
    f"{'Cardiovascular':45}"
    f"{qtd_cardiovascular:>15,}"
)
print(
    f"{'Respiratorio':45}"
    f"{qtd_respiratorio:>15,}"
)
print(
    f"{'Natureza acidentaria':45}"
    f"{qtd_acidentario:>15,}"
)
print("-" * 70)

erros_modelo = []

if total_registros != 185412:
    erros_modelo.append(
        "Quantidade de registros diferente de 185.412."
    )

if fato_sk_distintas != total_registros:
    erros_modelo.append(
        "A chave fato_sk nao e unica."
    )

if qtd_beneficios != total_registros:
    erros_modelo.append(
        "A soma de qtd_beneficios diverge do total."
    )

if qtd_saude_mental != 19572:
    erros_modelo.append(
        "Quantidade de saude mental diferente de 19.572."
    )

if qtd_osteomuscular != 36252:
    erros_modelo.append(
        "Quantidade osteomuscular diferente de 36.252."
    )

if qtd_cardiovascular <= 0:
    erros_modelo.append(
        "Quantidade cardiovascular nao positiva."
    )

if qtd_respiratorio <= 0:
    erros_modelo.append(
        "Quantidade respiratorio nao positiva."
    )

if qtd_acidentario != 13428:
    erros_modelo.append(
        "Quantidade acidentaria diferente de 13.428."
    )

if erros_modelo:
    raise ValueError(
        "Falha na validacao do modelo: "
        + " | ".join(erros_modelo)
    )

print("RESULTADO: OK")
print(
    "O modelo esta consistente para consumo analitico."
)
print("=" * 70)

## 5. Visão executiva

Os indicadores desta seção utilizam como universo os registros da tabela `fato_afastamentos`.

O denominador dos percentuais é a quantidade total de registros classificados como afastamento.

Os resultados não representam pessoas únicas ou taxas populacionais.

In [0]:
df_visao_executiva = (
    df_fato
    .agg(
        spark_sum(
            col("qtd_beneficios")
        ).alias(
            "total_afastamentos"
        ),
        spark_sum(
            col("ind_saude_mental")
        ).alias(
            "qtd_saude_mental"
        ),
        spark_sum(
            col("ind_osteomuscular")
        ).alias(
            "qtd_osteomuscular"
        ),
        spark_sum(
            col("ind_cardiovascular")
        ).alias(
            "qtd_cardiovascular"
        ),
        spark_sum(
            col("ind_respiratorio")
        ).alias(
            "qtd_respiratorio"
        ),
        spark_sum(
            col("ind_acidentario")
        ).alias(
            "qtd_acidentario"
        ),
        spark_sum(
            col("ind_cid_informado")
        ).alias(
            "qtd_cid_informado"
        ),
        avg(
            col("duracao_beneficio_dias")
        ).alias(
            "duracao_media_dias"
        ),
        avg(
            col("qt_sm_rmi")
        ).alias(
            "rmi_media_salarios_minimos"
        )
    )
    .withColumn(
        "pct_saude_mental",
        spark_round(
            col("qtd_saude_mental")
            / col("total_afastamentos")
            * lit(100),
            2
        )
    )
    .withColumn(
        "pct_osteomuscular",
        spark_round(
            col("qtd_osteomuscular")
            / col("total_afastamentos")
            * lit(100),
            2
        )
    )
    .withColumn(
        "pct_cardiovascular",
        spark_round(
            col("qtd_cardiovascular")
            / col("total_afastamentos")
            * lit(100),
            2
        )
    )
    .withColumn(
        "pct_respiratorio",
        spark_round(
            col("qtd_respiratorio")
            / col("total_afastamentos")
            * lit(100),
            2
        )
    )
    .withColumn(
        "pct_acidentario",
        spark_round(
            col("qtd_acidentario")
            / col("total_afastamentos")
            * lit(100),
            2
        )
    )
    .withColumn(
        "pct_cid_informado",
        spark_round(
            col("qtd_cid_informado")
            / col("total_afastamentos")
            * lit(100),
            2
        )
    )
    .withColumn(
        "duracao_media_dias",
        spark_round(
            col("duracao_media_dias"),
            2
        )
    )
    .withColumn(
        "rmi_media_salarios_minimos",
        spark_round(
            col("rmi_media_salarios_minimos"),
            2
        )
    )
)

display(df_visao_executiva)

In [0]:
linha_executiva = df_visao_executiva.first()

erros_visao_executiva = []

if linha_executiva["total_afastamentos"] != 185412:
    erros_visao_executiva.append(
        "Total de afastamentos inconsistente."
    )

if linha_executiva["qtd_saude_mental"] != 19572:
    erros_visao_executiva.append(
        "Quantidade de saude mental inconsistente."
    )

if linha_executiva["qtd_osteomuscular"] != 36252:
    erros_visao_executiva.append(
        "Quantidade osteomuscular inconsistente."
    )

if linha_executiva["qtd_cardiovascular"] <= 0:
    erros_visao_executiva.append(
        "Quantidade cardiovascular inconsistente."
    )

if linha_executiva["qtd_respiratorio"] <= 0:
    erros_visao_executiva.append(
        "Quantidade respiratorio inconsistente."
    )

if linha_executiva["qtd_acidentario"] != 13428:
    erros_visao_executiva.append(
        "Quantidade acidentaria inconsistente."
    )

if erros_visao_executiva:
    raise ValueError(
        "Falha na visao executiva: "
        + " | ".join(erros_visao_executiva)
    )

print("RESULTADO: OK")
print(
    "Os indicadores executivos correspondem "
    "aos valores validados no modelo."
)

### Leitura inicial dos indicadores

Na competência analisada, foram identificados 185.412 registros de benefícios classificados como afastamento.

Desse total:

- 19.572 registros, equivalentes a 10,56%, estão associados a CIDs do grupo de transtornos mentais e comportamentais;
- 36.252 registros, equivalentes a 19,55%, estão associados a CIDs do grupo de doenças osteomusculares;
- 10.312 registros, equivalentes a 5,56%, estão associados a CIDs do grupo de doenças cardiovasculares;
- 1.302 registros, equivalentes a 0,70%, estão associados a CIDs do grupo de doenças respiratórias;
- 13.428 registros, equivalentes a 7,24%, possuem natureza administrativa acidentária;
- 170.114 registros, equivalentes a 91,75%, possuem CID informado;
- a duração média entre benefícios com data de cessação disponível foi de 131,21 dias;
- a RMI média observada foi de 1,37 salários mínimos.

Os percentuais representam a distribuição interna dos registros de afastamento. Não representam pessoas únicas, trabalhadores únicos ou taxas populacionais.

In [0]:
df_fato_com_cid = (
    df_fato.alias("f")
    .join(
        df_dim_cid.alias("c"),
        col("f.cid_sk") == col("c.cid_sk"),
        "inner"
    )
)

df_duracao_por_grupo = (
    df_fato_com_cid
    .groupBy(
        col("c.cid_grupo").alias("cid_grupo"),
        col("c.cid_grupo_desc").alias(
            "cid_grupo_desc"
        )
    )
    .agg(
        spark_sum(
            col("f.qtd_beneficios")
        ).alias(
            "qtd_registros"
        ),
        avg(
            col("f.duracao_beneficio_dias")
        ).alias(
            "duracao_media_dias"
        )
    )
    .withColumn(
        "duracao_media_dias",
        spark_round(
            col("duracao_media_dias"),
            2
        )
    )
    .orderBy(
        col("qtd_registros").desc()
    )
)

display(df_duracao_por_grupo)